# Lab 6: Running Experiments

## Difficulty: Intermediate | ~45 min | Requires Labs 3 and 5

Learn how LangSmith experiments let you run a configuration against an entire dataset, score every example with evaluators, and compare two configurations side by side using experiment metadata.

In [ ]:
!pip install -qU "langsmith>=0.1.0" "langchain-core>=0.2.0" "langchain-openai>=0.1.0" "openai>=1.0.0" "python-dotenv>=1.0.0" "pydantic>=2.0.0"

## Cell 2: Load Environment and Initialize Clients

Loads API keys and sets up the LangSmith client (for dataset access and evaluation) and the OpenAI client (for the LLM-as-judge evaluator).

In [ ]:
import os
from dotenv import load_dotenv
from langsmith import Client
from openai import OpenAI

load_dotenv()
assert os.getenv("OPENROUTER_API_KEY"), "Missing OPENROUTER_API_KEY"
assert os.getenv("LANGSMITH_API_KEY"), "Missing LANGSMITH_API_KEY"

ls_client = Client()
judge_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)
print("Clients ready")

## Cell 3: Load the Lab 3 Dataset

Pulls the `product-reviews` dataset from Lab 3. Each example has a raw review as input and a structured `ProductReview` output (product, rating, sentiment).

In [ ]:
dataset = ls_client.read_dataset(dataset_name="product-reviews")
examples = list(ls_client.list_examples(dataset_id=dataset.id))
print(f"Loaded {len(examples)} examples from '{dataset.name}'")
for ex in examples[:3]:
    print(f"  Input: {ex.inputs}")
    print(f"  Output: {ex.outputs}\n")

## Cell 4: Define the Schema and Model

Re-establishes the structured-output agent from Lab 3/5. Both target functions will use this same model — the only difference is whether middleware preprocesses the input first.

In [ ]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

class ProductReview(BaseModel):
    product: str = Field(description="The exact product being reviewed")
    rating: int = Field(description="The star rating, from 1 to 5")
    sentiment: str = Field(description="positive, negative, or neutral")

model = ChatOpenAI(
    model="nvidia/nemotron-3.5-lightning:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0,
)
structured_model = model.with_structured_output(ProductReview)

## Cell 5: Define the Three Evaluators

The same three evaluators from Lab 5 — schema validation (heuristic), helpfulness scoring (LLM-as-judge), and guardrail compliance (custom). Re-defined here so this lab is self-contained.

In [ ]:
from langsmith.evaluation import evaluate
from pydantic import ValidationError

def schema_validator(run, example) -> dict:
    """Heuristic: check if output parses against ProductReview schema."""
    try:
        ProductReview(**run.outputs)
        return {"key": "schema_valid", "score": True}
    except ValidationError as e:
        return {"key": "schema_valid", "score": False, "comment": str(e)}

def helpfulness_judge(run, example) -> dict:
    """LLM-as-judge: score output helpfulness from 1-5."""
    import json
    prompt = f"""Score this extraction on a 1-5 helpfulness scale.
Review: {example.inputs['review_text']}
Extracted: {run.outputs}
Reference: {example.outputs}
Return ONLY JSON: {{"score": <int>, "reason": "<brief explanation>"}}"""
    try:
        response = judge_client.chat.completions.create(
            model="nvidia/nemotron-3-super-120b-a12b:free",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        result = json.loads(response.choices[0].message.content)
        return {"key": "helpfulness", "score": result["score"], "comment": result.get("reason", "")}
    except Exception as e:
        return {"key": "helpfulness", "score": None, "comment": f"Judge error: {e}"}

ALLOWED_SENTIMENTS = {"positive", "negative", "neutral"}

def guardrail_checker(run, example) -> dict:
    """Custom: enforce sentiment, rating, and product guardrails."""
    output = run.outputs
    violations = []
    if output.get("sentiment") not in ALLOWED_SENTIMENTS:
        violations.append(f"Invalid sentiment: {output.get('sentiment')}")
    if not (1 <= output.get("rating", 0) <= 5):
        violations.append(f"Rating out of range: {output.get('rating')}")
    if not output.get("product"):
        violations.append("Missing product name")
    return {
        "key": "guardrail_pass",
        "score": len(violations) == 0,
        "comment": "; ".join(violations) if violations else "All guardrails passed"
    }

print("Three evaluators defined: schema_validator, helpfulness_judge, guardrail_checker")

## Cell 6: Define Two Target Functions (With and Without Middleware)

The middleware is a `normalize_text` function that strips whitespace and collapses multiple spaces. Both targets use the same model — the only difference is whether this preprocessing step runs first.

In [ ]:
import re

def normalize_text(text: str) -> str:
    """Middleware: normalize review text before sending to the model."""
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def target_with_middleware(inputs: dict) -> dict:
    """Target with text-normalization middleware applied first."""
    normalized = normalize_text(inputs["review_text"])
    parsed = structured_model.invoke(normalized)
    return parsed.model_dump()

def target_without_middleware(inputs: dict) -> dict:
    """Target with no preprocessing — raw input goes straight to the model."""
    parsed = structured_model.invoke(inputs["review_text"])
    return parsed.model_dump()

print("Two targets defined: with middleware (normalize_text) and without")

## Cell 7: Run Experiment 1 — With Middleware

Run the target with middleware across every example. The `experiment_prefix` names the experiment in LangSmith, and `metadata` attaches key-value pairs for later filtering.

In [ ]:
results_with = evaluate(
    target_with_middleware,
    data="product-reviews",
    evaluators=[schema_validator, helpfulness_judge, guardrail_checker],
    experiment_prefix="lab6-with-middleware",
    metadata={"middleware": "text-normalization", "variant": "with"},
)
print(f"Experiment 'with middleware' complete: {len(results_with._results)} examples scored")

## Cell 8: Run Experiment 2 — Without Middleware

Same evaluators, same dataset, different target function. The only variable that changed is the middleware — this is what makes the comparison valid.

In [ ]:
results_without = evaluate(
    target_without_middleware,
    data="product-reviews",
    evaluators=[schema_validator, helpfulness_judge, guardrail_checker],
    experiment_prefix="lab6-without-middleware",
    metadata={"middleware": "none", "variant": "without"},
)
print(f"Experiment 'without middleware' complete: {len(results_without._results)} examples scored")

## Cell 9: Compare Results Across Experiments

Extract scores from both experiments and print a side-by-side comparison. This is the payoff of controlled experiments — a clear, data-driven view of what changed.

In [ ]:
def extract_scores(results):
    all_results = results._results
    schema = [r["evaluation_results"]["results"][0].score for r in all_results]
    helpfulness = [r["evaluation_results"]["results"][1].score for r in all_results]
    guardrail = [r["evaluation_results"]["results"][2].score for r in all_results]
    valid_help = [h for h in helpfulness if h is not None]
    h_avg = f"{sum(valid_help)/len(valid_help):.1f}/5" if valid_help else "N/A"
    return {
        "schema_pass": f"{sum(schema)}/{len(schema)} ({sum(schema)/len(schema)*100:.0f}%)",
        "guardrail_pass": f"{sum(guardrail)}/{len(guardrail)} ({sum(guardrail)/len(guardrail)*100:.0f}%)",
        "helpfulness_avg": h_avg,
    }

scores_with = extract_scores(results_with)
scores_without = extract_scores(results_without)

print("=== Experiment Comparison ===")
print('{:25}{:18}{:18}'.format('', 'With Middleware', 'Without Middleware'))
print('{:25}{:18}{:18}'.format('Schema Valid:', scores_with['schema_pass'], scores_without['schema_pass']))
print('{:25}{:18}{:18}'.format('Guardrail Pass:', scores_with['guardrail_pass'], scores_without['guardrail_pass']))
print('{:25}{:18}{:18}'.format('Helpfulness Avg:', scores_with['helpfulness_avg'], scores_without['helpfulness_avg']))

## Optional Exercise

Add a third middleware variant that uppercases all review text before sending to the model. Define a new target function `target_with_uppercase_middleware`, run it as a third experiment with `experiment_prefix="lab6-uppercase-middleware"` and `metadata={"middleware": "uppercase", "variant": "uppercase"}`, then extend the comparison table to show all three configurations side by side.